# ROAD-SHIELD: real-data detector training
Trains one YOLO detector for potholes (incl. water-filled) and cracks on human-labelled data only, with duplicate-clustered splits, a selection rule fixed before test scoring, and measured-only metrics. Runtime: GPU (T4).

In [ ]:
!pip -q install ultralytics opencv-python-headless pyyaml
!git clone https://github.com/udbhav968-creator/SIH_PROJECT.git repo || true
%cd repo/sih_project

Put each real labelled source (YOLO format) under `/content/data/<source>/images|labels` and write `/content/data/sources.json` with url, licence, `human_labelled: true` and a class_map. See the docstring of `training/train_detectors_real.py`.

In [ ]:
# Upload kaggle.json first (Files panel), then:
!mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kagglehub pyyaml
# Replace with the real pothole/crack datasets you chose; classes must map to our 4 targets:
!python training/prepare_kaggle_detection_data.py --out /content/data --dataset OWNER/SLUG:pothole
# Then edit /content/data/sources.json: set the real licence for each source.

In [ ]:
!python training/train_detectors_real.py --data-root /content/data --out /content/out --epochs 100

In [ ]:
# Optional: add the 5 hand-annotated sample photos as extra training examples (pothole boxes only)
!python training/export_samples_as_source.py --data-root /content/data
# then set their real licence in /content/data/sources.json before training

In [ ]:
import json; print(json.dumps(json.load(open('/content/out/detector_metrics.json')), indent=2)[:4000])

Copy `/content/out/train/weights/best.onnx` to `checkpoints/damage_rdd2022_india.onnx` and `detector_metrics.json` to `checkpoints/damage_rdd2022_india.json`, then run `scripts/verify_rdd_detector.py`. The pipeline will not draw boxes until that script sets `artefact_check` and `deployment_check`.